# Forward-Backward RL Beyond Imitation & World-Model Acceleration — GPU Experiment Suite (B01–B05)

This self-contained notebook executes the five verification and ablation suites (`B01` through `B05`) on a Google Colab GPU (`L4` / `A100` / `T4`) using PyTorch and MuJoCo, and saves `colab_gpu_results.json`.

In [ ]:
!pip install -q torch scipy mujoco numpy pytest
import torch, mujoco
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
print('MuJoCo:', mujoco.__version__)

In [ ]:
%%writefile fb_core.py
"""Core Forward-Backward (FB) + Low-Level Actor + Task-Q + Planner modules.

Implements the exact mathematical and engineering contracts from Baselines 01-05:
- Spherical backward map B(m) with radius sqrt(d_z)
- Forward successor ensemble F_{1,2}(o_F, a, z) with scalar inner-product order
- Causal EMA Gram matrix tracker Sigma_B and damped Cholesky whitening W_lambda
- Strict UFO update contract: single-pass BN update on (obs, m_current) and
  (next_obs, m_next), BN freeze, 50/50 sample_mixed_z relabeling without
  mutating behavior_z
- Full-prefix zero-hidden RNN burn-in for recurrent planner P and task critic Q
"""

from __future__ import annotations
import math
from dataclasses import dataclass
from typing import Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


def project_to_sphere(x: torch.Tensor, d_z: Optional[int] = None, eps: float = 1e-8) -> torch.Tensor:
    """Projects last-dimension vectors onto the sphere of radius sqrt(d_z)."""
    dim = x.shape[-1] if d_z is None else d_z
    radius = math.sqrt(float(dim))
    norm = torch.linalg.vector_norm(x, ord=2, dim=-1, keepdim=True)
    return radius * x / (norm + eps)


class RunningMeanStdNormalizer(nn.Module):
    """Causal running mean/std normalizer with explicit update freeze control."""

    def __init__(self, shape: Tuple[int, ...], eps: float = 1e-5, clip: float = 5.0):
        super().__init__()
        self.eps = eps
        self.clip = clip
        self.register_buffer("mean", torch.zeros(shape, dtype=torch.float32))
        self.register_buffer("var", torch.ones(shape, dtype=torch.float32))
        self.register_buffer("count", torch.tensor(1e-4, dtype=torch.float64))
        self. register_buffer("update_calls", torch.tensor(0, dtype=torch.long))

    @torch.no_grad()
    def update_stats(self, x: torch.Tensor) -> None:
        if not self.training:
            raise RuntimeError("Attempted to update RunningMeanStdNormalizer while frozen (eval mode).")
        if not torch.isfinite(x).all():
            raise ValueError("Cannot update normalizer with non-finite tensor (invalid physics).")
        flat = x.reshape(-1, x.shape[-1]).to(dtype=torch.float64)
        batch_mean = flat.mean(dim=0)
        batch_var = flat.var(dim=0, unbiased=False)
        batch_count = float(flat.shape[0])

        delta = batch_mean - self.mean.to(torch.float64)
        tot_count = self.count + batch_count
        new_mean = self.mean.to(torch.float64) + delta * (batch_count / tot_count)
        m_a = self.var.to(torch.float64) * self.count
        m_b = batch_var * batch_count
        m2 = m_a + m_b + delta.pow(2) * self.count * batch_count / tot_count
        new_var = m2 / tot_count

        self.mean.copy_(new_mean.to(torch.float32))
        self.var.copy_(new_var.to(torch.float32))
        self.count.copy_(tot_count)
        self.update_calls.add_(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        std = torch.sqrt(self.var + self.eps)
        normed = (x - self.mean) / std
        return torch.clamp(normed, -self.clip, self.clip)


class EMAWhiteningTracker(nn.Module):
    """Tracks EMA second-moment matrix Sigma_B = E[B(x)B(x)^T] and damped inverse."""

    def __init__(self, d_z: int, beta: float = 0.05, lambda_reg: float = 1e-2):
        super().__init__()
        self.d_z = d_z
        self.beta = beta
        self.lambda_reg = lambda_reg
        self.register_buffer("sigma_b", torch.eye(d_z, dtype=torch.float32))
        self.register_buffer("num_updates", torch.tensor(0, dtype=torch.long))

    @torch.no_grad()
    def update(self, b_batch: torch.Tensor) -> None:
        # b_batch: [N, d_z] on sphere of radius sqrt(d_z)
        n = b_batch.shape[0]
        gram = (b_batch.transpose(0, 1) @ b_batch) / float(max(n, 1))
        gram = 0.5 * (gram + gram.transpose(0, 1))
        if int(self.num_updates.item()) == 0:
            self.sigma_b.copy_(gram)
        else:
            self.sigma_b.lerp_(gram, self.beta)
        self.num_updates.add_(1)

    @torch.no_grad()
    def spectral_diagnostics(self, mat: Optional[torch.Tensor] = None) -> Dict[str, float]:
        target = self.sigma_b if mat is None else mat
        sym = 0.5 * (target + target.transpose(0, 1))
        evals = torch.linalg.eigvalsh(sym).clamp(min=1e-12)
        l_max = float(evals[-1].item())
        l_min = float(evals[0].item())
        cond = l_max / max(l_min, 1e-12)
        probs = evals / evals.sum()
        entropy = -torch.sum(probs * torch.log(probs + 1e-12))
        erank = float(torch.exp(entropy).item())
        off_diag_frob = float(torch.linalg.matrix_norm(sym - torch.diag(torch.diag(sym)), ord="fro").item())
        return {
            "lambda_max": l_max,
            "lambda_min": l_min,
            "condition_number": cond,
            "effective_rank": erank,
            "trace": float(evals.sum().item()),
            "off_diag_frobenius": off_diag_frob,
        }

    def whiten_vector(self, v: torch.Tensor, lambda_reg: Optional[float] = None) -> torch.Tensor:
        """Computes (Sigma_B + lambda_reg * I)^{-1} v via Cholesky decomposition."""
        reg = self.lambda_reg if lambda_reg is None else lambda_reg
        eye = torch.eye(self.d_z, device=v.device, dtype=v.dtype)
        damped = 0.5 * (self.sigma_b + self.sigma_b.transpose(0, 1)) + reg * eye
        chol = torch.linalg.cholesky(damped)
        # Solve damped @ x = v^T
        sol = torch.cholesky_solve(v.transpose(-2, -1), chol)
        return sol.transpose(-2, -1)

    def intrinsic_reward(self, b_next: torch.Tensor, z: torch.Tensor, whitened: bool = True) -> torch.Tensor:
        """Computes one-step intrinsic reward r_z(y) = B(y)^T Sigma_B^{-1} z or raw B(y)^T z."""
        if whitened:
            phi_next = self.whiten_vector(b_next)
            return (phi_next * z).sum(dim=-1)
        return (b_next * z).sum(dim=-1)


class BackwardMapB(nn.Module):
    """Single-frame motion encoder B_omega(m) -> S^{d_z - 1}(sqrt(d_z))."""

    def __init__(self, motion_dim: int, d_z: int = 256, hidden_dim: int = 256):
        super().__init__()
        self.motion_dim = motion_dim
        self.d_z = d_z
        self.net = nn.Sequential(
            nn.Linear(motion_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, d_z),
        )

    def forward(self, m_normed: torch.Tensor) -> torch.Tensor:
        raw = self.net(m_normed)
        return project_to_sphere(raw, self.d_z)


class ForwardHeadF(nn.Module):
    """Two-branch forward successor network F(o_F, a, z) -> R^{d_z}."""

    def __init__(self, obs_f_dim: int, action_dim: int, d_z: int = 256, hidden_dim: int = 256):
        super().__init__()
        self.sa_branch = nn.Sequential(
            nn.Linear(obs_f_dim + action_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
        )
        self.sz_branch = nn.Sequential(
            nn.Linear(obs_f_dim + d_z, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
        )
        self.trunk = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, d_z),
        )

    def forward(self, obs_f: torch.Tensor, action: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        h_sa = self.sa_branch(torch.cat([obs_f, action], dim=-1))
        h_sz = self.sz_branch(torch.cat([obs_f, z], dim=-1))
        return self.trunk(torch.cat([h_sa, h_sz], dim=-1))


class ForwardEnsembleF(nn.Module):
    """Twin forward successor heads F_1, F_2 enforcing scalar inner-product order."""

    def __init__(self, obs_f_dim: int, action_dim: int, d_z: int = 256, hidden_dim: int = 256):
        super().__init__()
        self.f1 = ForwardHeadF(obs_f_dim, action_dim, d_z, hidden_dim)
        self.f2 = ForwardHeadF(obs_f_dim, action_dim, d_z, hidden_dim)

    def forward(self, obs_f: torch.Tensor, action: torch.Tensor, z: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.f1(obs_f, action, z), self.f2(obs_f, action, z)

    def conservative_scalar_value(self, obs_f: torch.Tensor, action: torch.Tensor, w: torch.Tensor, z: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Computes min_{k=1,2} (F_k(obs_f, action, z)^T w) AFTER scalar inner products."""
        z_cond = w if z is None else z
        f1_out, f2_out = self.forward(obs_f, action, z_cond)
        q1 = (f1_out * w).sum(dim=-1)
        q2 = (f2_out * w).sum(dim=-1)
        return torch.minimum(q1, q2)


class ConditionedLowLevelActor(nn.Module):
    """Low-level conditioned actor pi_theta(o_low, z) -> [-1, 1]^{action_dim}."""

    def __init__(self, obs_low_dim: int, action_dim: int, d_z: int = 256, hidden_dim: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_low_dim + d_z, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, obs_low: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        return torch.tanh(self.net(torch.cat([obs_low, z], dim=-1)))


class RecurrentLatentPlanner(nn.Module):
    """High-level recurrent planner P_psi(H_plan) -> S^{d_z - 1}(sqrt(d_z)) with zero-hidden burn-in."""

    def __init__(self, obs_plan_dim: int, d_z: int = 256, hidden_dim: int = 128):
        super().__init__()
        self.d_z = d_z
        self.hidden_dim = hidden_dim
        self.rnn = nn.LSTM(obs_plan_dim, hidden_dim, num_layers=2, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, d_z),
        )

    def forward_sequence(
        self,
        obs_seq: torch.Tensor,
        burn_in_len: int = 0,
    ) -> torch.Tensor:
        """Runs full-prefix zero-hidden burn-in on [:, :burn_in_len] under no_grad, then BPTT."""
        bsz = obs_seq.shape[0]
        device = obs_seq.device
        h0 = torch.zeros(2, bsz, self.hidden_dim, device=device, dtype=obs_seq.dtype)
        c0 = torch.zeros(2, bsz, self.hidden_dim, device=device, dtype=obs_seq.dtype)

        if burn_in_len > 0:
            with torch.no_grad():
                _, (h_burn, c_burn) = self.rnn(obs_seq[:, :burn_in_len, :], (h0, c0))
            h_init = h_burn.detach()
            c_init = c_burn.detach()
            active_seq = obs_seq[:, burn_in_len:, :]
        else:
            h_init, c_init = h0, c0
            active_seq = obs_seq

        rnn_out, _ = self.rnn(active_seq, (h_init, c_init))
        raw_z = self.head(rnn_out)
        return project_to_sphere(raw_z, self.d_z)


class RecurrentTaskTwinCritic(nn.Module):
    """Recurrent Twin Task Critic Q_{1,2}(H_Q, cmd) supporting either macro-z or step-action cmd."""

    def __init__(self, obs_q_dim: int, cmd_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.rnn1 = nn.LSTM(obs_q_dim, hidden_dim, num_layers=2, batch_first=True)
        self.rnn2 = nn.LSTM(obs_q_dim, hidden_dim, num_layers=2, batch_first=True)
        self.q1_head = nn.Sequential(
            nn.Linear(hidden_dim + cmd_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.q2_head = nn.Sequential(
            nn.Linear(hidden_dim + cmd_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward_sequence(
        self,
        obs_q_seq: torch.Tensor,
        cmd_seq: torch.Tensor,
        burn_in_len: int = 0,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        bsz = obs_q_seq.shape[0]
        device = obs_q_seq.device
        h0 = torch.zeros(2, bsz, self.hidden_dim, device=device, dtype=obs_q_seq.dtype)
        c0 = torch.zeros(2, bsz, self.hidden_dim, device=device, dtype=obs_q_seq.dtype)

        if burn_in_len > 0:
            with torch.no_grad():
                _, (h1_b, c1_b) = self.rnn1(obs_q_seq[:, :burn_in_len, :], (h0, c0))
                _, (h2_b, c2_b) = self.rnn2(obs_q_seq[:, :burn_in_len, :], (h0, c0))
            s1 = (h1_b.detach(), c1_b.detach())
            s2 = (h2_b.detach(), c2_b.detach())
            active_obs = obs_q_seq[:, burn_in_len:, :]
            active_cmd = cmd_seq[:, burn_in_len:, :]
        else:
            s1 = (h0, c0)
            s2 = (h0, c0)
            active_obs = obs_q_seq
            active_cmd = cmd_seq

        out1, _ = self.rnn1(active_obs, s1)
        out2, _ = self.rnn2(active_obs, s2)
        q1 = self.q1_head(torch.cat([out1, active_cmd], dim=-1)).squeeze(-1)
        q2 = self.q2_head(torch.cat([out2, active_cmd], dim=-1)).squeeze(-1)
        return q1, q2


def compute_orthonormalization_loss(b_batch: torch.Tensor) -> torch.Tensor:
    """Computes standard FB orthonormalization regularizer on B(x)."""
    n = b_batch.shape[0]
    gram = b_batch @ b_batch.transpose(0, 1)
    diag = torch.diag(gram)
    off_diag_sq = (gram.pow(2).sum() - diag.pow(2).sum()) / float(max(n * (n - 1), 1))
    pull = diag.mean()
    return 0.5 * off_diag_sq - pull


@torch.no_grad()
def sample_ufo_mixed_z(
    b_net: BackwardMapB,
    m_next_normed: torch.Tensor,
    goal_ratio: float = 0.5,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Samples 50% online-B permuted m_next goals and 50% random sphere latents under frozen BN."""
    n = m_next_normed.shape[0]
    d_z = b_net.d_z
    device = m_next_normed.device
    z_rand = project_to_sphere(torch.randn(n, d_z, device=device, dtype=m_next_normed.dtype), d_z)
    perm = torch.randperm(n, device=device)
    z_goal = b_net(m_next_normed[perm])
    mask = (torch.rand(n, 1, device=device) < goal_ratio)
    z_update = torch.where(mask, z_goal, z_rand)
    return z_update, perm


In [ ]:
%%writefile physics_env.py
"""MuJoCo and Differentiable Batched Contact Physics Environments for B01-B05.

Provides:
1. MuJoCoObstacleTraversalR21Env:
   - Real MuJoCo 3.17 rigid-body simulation with 4 physics substeps (dt_physics=0.005s,
     dt_control=0.02s) and a configurable 3D box obstacle (h in {0.10..0.70} m, D=0.356 m).
   - Implements the exact R2.1 4-substep terminal boundary state machine:
     strict priority `invalid` > `severe_penetration` (> 0.03 m) > `success` > `timeout`.
2. DifferentiableContactPlant:
   - Batched PyTorch GPU/CPU physics plant with unilateral box contact, friction cones,
     and ball-impulse dynamics for controlled pathwise gradient vs FB derivative ablations.
"""

from __future__ import annotations
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import math
import mujoco
import numpy as np
import torch
import torch.nn as nn


@dataclass
class SubstepSnapshot:
    substep_index: int
    max_penetration_m: float
    finite_state: bool
    finite_contacts: bool
    ncon: int
    nconmax: int
    solver_diverged: bool
    schema_ok: bool


@dataclass
class ControlStepOutcome:
    status: str  # "continue", "severe_penetration", "invalid", "success", "timeout"
    terminated: bool
    truncated: bool
    commit_to_replay: bool
    update_normalizers: bool
    increment_step_counter: bool
    halt_run_or_panel: bool
    phi_next: float
    offending_substep: Optional[int]
    diagnostic_reason: str
    substep_trace: List[SubstepSnapshot] = field(default_factory=list)


def evaluate_r21_substep_state_machine(
    substeps: List[SubstepSnapshot],
    final_state_finite: bool,
    final_schema_ok: bool,
    raw_phi_next: float,
    is_success: bool = False,
    is_timeout: bool = False,
    penetration_limit_m: float = 0.03,
) -> ControlStepOutcome:
    """Evaluates all 4 physics substeps + post-step final state under strict R2.1 dominance.

    Dominance order:
      1. `invalid` (strictly overrides everything across all 4 substeps + final check)
      2. `severe_penetration` (latched on first substep > 0.03 m if all substeps are valid;
         sets phi_next = 0.0, terminated = True, commits once)
      3. `success` (terminated = True, phi_next = raw_phi_next)
      4. `timeout` (truncated = True, terminated = False, phi_next = raw_phi_next)
      5. `continue`
    """
    if len(substeps) != 4:
        return ControlStepOutcome(
            status="invalid",
            terminated=False,
            truncated=False,
            commit_to_replay=False,
            update_normalizers=False,
            increment_step_counter=False,
            halt_run_or_panel=True,
            phi_next=0.0,
            offending_substep=None,
            diagnostic_reason=f"incomplete_substep_count:{len(substeps)}",
            substep_trace=substeps,
        )

    latched_penetration_substep: Optional[int] = None
    # Crucial R2.1 invariant: NEVER break early on penetration; inspect all 4 substeps!
    for snap in substeps:
        is_corrupt = (
            (not snap.finite_state)
            or (not snap.finite_contacts)
            or (snap.ncon >= snap.nconmax)
            or snap.solver_diverged
            or (not snap.schema_ok)
            or math.isnan(snap.max_penetration_m)
            or math.isinf(snap.max_penetration_m)
        )
        if is_corrupt:
            return ControlStepOutcome(
                status="invalid",
                terminated=False,
                truncated=False,
                commit_to_replay=False,
                update_normalizers=False,
                increment_step_counter=False,
                halt_run_or_panel=True,
                phi_next=0.0,
                offending_substep=snap.substep_index,
                diagnostic_reason=f"substep_{snap.substep_index}_corruption",
                substep_trace=substeps,
            )
        if snap.max_penetration_m > penetration_limit_m and latched_penetration_substep is None:
            latched_penetration_substep = snap.substep_index

    if (not final_state_finite) or (not final_schema_ok) or math.isnan(raw_phi_next) or math.isinf(raw_phi_next):
        return ControlStepOutcome(
            status="invalid",
            terminated=False,
            truncated=False,
            commit_to_replay=False,
            update_normalizers=False,
            increment_step_counter=False,
            halt_run_or_panel=True,
            phi_next=0.0,
            offending_substep=4,
            diagnostic_reason="post_step_final_state_corruption",
            substep_trace=substeps,
        )

    if latched_penetration_substep is not None:
        return ControlStepOutcome(
            status="severe_penetration",
            terminated=True,
            truncated=False,
            commit_to_replay=True,
            update_normalizers=True,
            increment_step_counter=True,
            halt_run_or_panel=False,
            phi_next=0.0,
            offending_substep=latched_penetration_substep,
            diagnostic_reason=f"severe_penetration_substep_{latched_penetration_substep}",
            substep_trace=substeps,
        )

    if is_success:
        return ControlStepOutcome(
            status="success",
            terminated=True,
            truncated=False,
            commit_to_replay=True,
            update_normalizers=True,
            increment_step_counter=True,
            halt_run_or_panel=False,
            phi_next=float(raw_phi_next),
            offending_substep=None,
            diagnostic_reason="traversal_success",
            substep_trace=substeps,
        )

    if is_timeout:
        return ControlStepOutcome(
            status="timeout",
            terminated=False,
            truncated=True,
            commit_to_replay=True,
            update_normalizers=True,
            increment_step_counter=True,
            halt_run_or_panel=False,
            phi_next=float(raw_phi_next),
            offending_substep=None,
            diagnostic_reason="episode_timeout",
            substep_trace=substeps,
        )

    return ControlStepOutcome(
        status="continue",
        terminated=False,
        truncated=False,
        commit_to_replay=True,
        update_normalizers=True,
        increment_step_counter=True,
        halt_run_or_panel=False,
        phi_next=float(raw_phi_next),
        offending_substep=None,
        diagnostic_reason="nominal",
        substep_trace=substeps,
    )


class MuJoCoObstacleTraversalR21Env:
    """Real MuJoCo 3.17 multi-link obstacle traversal environment with 4-substep R2.1 boundary check."""

    def __init__(self, obstacle_height: float = 0.20, nconmax: int = 64):
        self.obstacle_height = obstacle_height
        self.nconmax = nconmax
        xml = f"""
        <mujoco model="obstacle_traversal_r21">
          <option timestep="0.005" gravity="0 0 -9.81" integrator="Euler"/>
          <size nconmax="{nconmax}"/>
          <worldbody>
            <geom name="floor" type="plane" size="10 10 0.1" rgba="0.8 0.8 0.8 1"/>
            <geom name="obstacle_box" type="box" pos="0.6 0 {obstacle_height / 2.0:.4f}"
                  size="0.1779775 5.0 {obstacle_height / 2.0:.4f}" rgba="0.3 0.5 0.7 1"/>
            <body name="torso_link" pos="0 0 0.85">
              <freejoint name="root"/>
              <geom name="torso_geom" type="capsule" size="0.08 0.18" mass="12.0"/>
              <body name="left_hand" pos="0.22 0.18 -0.05">
                <joint name="left_shoulder_pitch" type="hinge" axis="0 1 0" range="-2.5 2.5"/>
                <geom name="left_hand_geom" type="sphere" size="0.045" mass="1.0"/>
              </body>
              <body name="right_hand" pos="0.22 -0.18 -0.05">
                <joint name="right_shoulder_pitch" type="hinge" axis="0 1 0" range="-2.5 2.5"/>
                <geom name="right_hand_geom" type="sphere" size="0.045" mass="1.0"/>
              </body>
              <body name="left_leg" pos="0 0.11 -0.25">
                <joint name="left_hip_pitch" type="hinge" axis="0 1 0" range="-2.0 2.0"/>
                <geom name="left_thigh_geom" type="capsule" size="0.05 0.18" mass="3.5"/>
                <body name="left_shin" pos="0 0 -0.35">
                  <joint name="left_knee" type="hinge" axis="0 1 0" range="-2.2 0.2"/>
                  <geom name="left_foot_geom" type="sphere" size="0.05" pos="0 0 -0.20" mass="1.5"/>
                </body>
              </body>
              <body name="right_leg" pos="0 -0.11 -0.25">
                <joint name="right_hip_pitch" type="hinge" axis="0 1 0" range="-2.0 2.0"/>
                <geom name="right_thigh_geom" type="capsule" size="0.05 0.18" mass="3.5"/>
                <body name="right_shin" pos="0 0 -0.35">
                  <joint name="right_knee" type="hinge" axis="0 1 0" range="-2.2 0.2"/>
                  <geom name="right_foot_geom" type="sphere" size="0.05" pos="0 0 -0.20" mass="1.5"/>
                </body>
              </body>
            </body>
          </worldbody>
          <actuator>
            <motor joint="left_shoulder_pitch" ctrlrange="-80 80" gear="1"/>
            <motor joint="right_shoulder_pitch" ctrlrange="-80 80" gear="1"/>
            <motor joint="left_hip_pitch" ctrlrange="-120 120" gear="1"/>
            <motor joint="left_knee" ctrlrange="-120 120" gear="1"/>
            <motor joint="right_hip_pitch" ctrlrange="-120 120" gear="1"/>
            <motor joint="right_knee" ctrlrange="-120 120" gear="1"/>
          </actuator>
        </mujoco>
        """
        self.model = mujoco.MjModel.from_xml_string(xml)
        self.data = mujoco.MjData(self.model)
        self.step_count = 0

    def reset(self, seed: int = 0) -> np.ndarray:
        rng = np.random.default_rng(seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[0] = rng.uniform(-0.02, 0.02)
        self.data.qpos[2] = 0.85
        self.data.qpos[3] = 1.0
        mujoco.mj_forward(self.model, self.data)
        self.step_count = 0
        return self.get_motion_descriptor_229()

    def get_motion_descriptor_229(self) -> np.ndarray:
        """Constructs a 229-dim root-relative frame descriptor matching Baseline 05 schema."""
        vec = np.zeros(229, dtype=np.float32)
        qpos = self.data.qpos
        qvel = self.data.qvel
        vec[0:3] = qvel[0:3]
        vec[3:6] = qvel[3:6]
        vec[6:9] = np.array([0.0, 0.0, -1.0], dtype=np.float32)
        n_j = min(len(qpos) - 7, 29)
        vec[9 : 9 + n_j] = qpos[7 : 7 + n_j]
        vec[38 : 38 + n_j] = qvel[6 : 6 + n_j]
        # Populate link relative positions and 6D rotations from xpos/xmat
        offset = 67
        for b_idx in range(1, min(self.model.nbody, 19)):
            rel_pos = self.data.xpos[b_idx] - self.data.xpos[1]
            xmat = self.data.xmat[b_idx].reshape(3, 3)
            rot6d = np.concatenate([xmat[:, 0], xmat[:, 1]])
            if offset + 9 <= 229:
                vec[offset : offset + 3] = rel_pos
                vec[offset + 3 : offset + 9] = rot6d
                offset += 9
        return vec

    def step_control(
        self,
        action_29: np.ndarray,
        inject_penetration_substep: Optional[int] = None,
        inject_nan_substep: Optional[int] = None,
    ) -> ControlStepOutcome:
        """Executes 4 physics substeps (4 x 0.005 s = 0.02 s) and runs R2.1 boundary check."""
        self.data.ctrl[:] = np.clip(action_29[: self.model.nu], -1.0, 1.0) * 60.0
        substeps: List[SubstepSnapshot] = []

        for j in range(1, 5):
            mujoco.mj_step(self.model, self.data)
            max_pen = 0.0
            for c_idx in range(self.data.ncon):
                dist = float(self.data.contact[c_idx].dist)
                if dist < 0.0:
                    max_pen = max(max_pen, -dist)
            if inject_penetration_substep == j:
                max_pen = max(max_pen, 0.045)
            if inject_nan_substep == j:
                self.data.qvel[0] = np.nan

            finite_s = bool(np.isfinite(self.data.qpos).all() and np.isfinite(self.data.qvel).all())
            finite_c = True
            for c_idx in range(self.data.ncon):
                if not np.isfinite(self.data.contact[c_idx].dist):
                    finite_c = False
            solver_div = bool(np.max(np.abs(self.data.qacc)) > 1e7) if finite_s else True
            substeps.append(
                SubstepSnapshot(
                    substep_index=j,
                    max_penetration_m=max_pen,
                    finite_state=finite_s,
                    finite_contacts=finite_c,
                    ncon=int(self.data.ncon),
                    nconmax=self.nconmax,
                    solver_diverged=solver_div,
                    schema_ok=True,
                )
            )

        final_finite = bool(np.isfinite(self.data.qpos).all() and np.isfinite(self.data.qvel).all())
        raw_phi = float(self.data.qpos[0] + 0.5 * self.data.qpos[2]) if final_finite else float("nan")
        is_succ = bool(final_finite and self.data.qpos[0] > 0.95 and self.data.qpos[2] > 0.70)
        outcome = evaluate_r21_substep_state_machine(
            substeps=substeps,
            final_state_finite=final_finite,
            final_schema_ok=True,
            raw_phi_next=raw_phi,
            is_success=is_succ,
            is_timeout=(self.step_count + 1 >= 600),
        )
        if outcome.increment_step_counter:
            self.step_count += 1
        return outcome


class DifferentiableContactPlant(nn.Module):
    """Batched differentiable contact-rich plant for GPU/CPU pathwise vs FB gradient experiments."""

    def __init__(self, state_dim: int = 32, action_dim: int = 12, motion_dim: int = 229, dt: float = 0.02):
        super().__init__()
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.motion_dim = motion_dim
        self.dt = dt
        # Fixed physical projection matrices (seeded for reproducibility)
        gen = torch.Generator().manual_seed(20261023)
        self.register_buffer("b_ctrl", torch.randn(state_dim // 2, action_dim, generator=gen) * 0.35)
        self.register_buffer("k_spring", torch.randn(state_dim // 2, state_dim // 2, generator=gen) * 0.15)
        self.register_buffer("kinematic_lift", torch.randn(motion_dim, state_dim, generator=gen) / math.sqrt(state_dim))

    def step(
        self,
        s: torch.Tensor,
        a: torch.Tensor,
        obstacle_h: Optional[torch.Tensor] = None,
        contact_stiffness: float = 25.0,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Advances batched state s=[q, v] by dt with smooth unilateral contact impulse."""
        half = self.state_dim // 2
        q = s[..., :half]
        v = s[..., half:]
        h_val = 0.30 if obstacle_h is None else obstacle_h
        # Unilateral box contact barrier around q[..., 0] (horizontal) and q[..., 1] (vertical clearance)
        penetration = torch.relu(h_val - q[..., 1:2]) * torch.sigmoid(10.0 * (0.4 - torch.abs(q[..., 0:1] - 0.5)))
        contact_force = contact_stiffness * penetration - 1.5 * v[..., 1:2] * (penetration > 0).to(s.dtype)

        acc = (a @ self.b_ctrl.transpose(0, 1)) - 0.8 * v - 0.25 * (q @ self.k_spring.transpose(0, 1))
        acc = acc.clone()
        acc[..., 1:2] = acc[..., 1:2] + contact_force

        v_next = torch.clamp(v + self.dt * acc, -8.0, 8.0)
        q_next = torch.clamp(q + self.dt * v_next, -4.0, 4.0)
        s_next = torch.cat([q_next, v_next], dim=-1)
        m_next = torch.tanh(s_next @ self.kinematic_lift.transpose(0, 1))
        return s_next, m_next


In [ ]:
%%writefile run_b01_gram_spike.py
"""Experiment Suite B01: FB Density Factorization, Gram Conditioning, and Intrinsic Reward Identifiability.

Tests:
1. Spectral conditioning of Sigma_B = E[B(x)B(x)^T] under anisotropic replay distributions
   for d_z in {100, 256} and motion_dim in {229, 277} despite exact spherical norm ||B(x)||_2 = sqrt(d_z).
2. Exact cancellation of Sigma_B in zero-shot linear reward readout z_r = E[B(x)r(x)] = Sigma_B lambda
   vs non-cancellation in single-state goal encoding z_g = B(g) and one-step intrinsic reward r_z(y).
3. Reduction of intrinsic reward MSE and improvement of policy-gradient cosine alignment via
   causal EMA damped Cholesky whitening W_lambda = (Sigma_B_hat + lambda_reg I)^{-1}.
"""

from __future__ import annotations
import json
import math
import time
from typing import Dict, Any
import torch
from fb_core import BackwardMapB, EMAWhiteningTracker, project_to_sphere


def run_b01_experiment(device_str: str = "auto", seed: int = 20261023) -> Dict[str, Any]:
    device = torch.device("cuda" if (device_str == "auto" and torch.cuda.is_available()) else ("cpu" if device_str == "auto" else device_str))
    torch.manual_seed(seed)
    t0 = time.time()
    results_by_dim = {}

    for d_z, m_dim in [(100, 277), (256, 229)]:
        # Construct an anisotropic latent replay distribution on S^{d_z-1}(sqrt(d_z))
        # simulating early online training where standing/falling transitions dominate
        n_samples = 4096
        raw_coords = torch.randn(n_samples, d_z, device=device)
        # Induce realistic spectral decay across dimensions
        decay = torch.exp(-torch.linspace(0.0, 2.8, d_z, device=device))
        b_samples = project_to_sphere(raw_coords * decay.unsqueeze(0), d_z)

        # True population Gram matrix on the batch
        sigma_true = (b_samples.transpose(0, 1) @ b_samples) / float(n_samples)
        tracker = EMAWhiteningTracker(d_z=d_z, beta=0.10, lambda_reg=1e-2).to(device)
        for chunk in torch.chunk(b_samples, 16, dim=0):
            tracker.update(chunk)

        spec = tracker.spectral_diagnostics(sigma_true)

        # Ground-truth successor expectation E[sum_t gamma^t B(s_{t+1})] = Sigma_B F(s,a,z)
        # Let Q_B(s,a,z) = Sigma_true @ F_true(s,a,z). Suppose the true target direction is z.
        # Under exact FB convergence, Sigma_true F(s,a,z) = z  <=>  F(s,a,z) = Sigma_true^{-1} z.
        n_queries = 512
        z_queries = project_to_sphere(torch.randn(n_queries, d_z, device=device), d_z)
        eye = torch.eye(d_z, device=device)
        sigma_inv_true = torch.linalg.inv(sigma_true + 1e-5 * eye)

        # 1. Linear task reward r(x) = B(x)^T lambda_true:
        #    True task Q_r = lambda_true^T Q_B. Zero-shot z_r = E[B(x)r(x)] = Sigma_true lambda_true.
        #    Since F = Sigma_true^{-1} Q_B, F^T z_r = Q_B^T Sigma_true^{-1} Sigma_true lambda_true = Q_r!
        lambda_true = torch.randn(d_z, 1, device=device)
        q_b = torch.randn(n_queries, d_z, device=device)
        f_converged = torch.linalg.solve(sigma_true.to(torch.float64), q_b.transpose(0, 1).to(torch.float64)).transpose(0, 1).to(torch.float32)
        q_r_exact = (q_b @ lambda_true).squeeze(-1)
        z_r = sigma_true @ lambda_true
        q_r_readout = (f_converged @ z_r).squeeze(-1)
        linear_readout_max_err = float(torch.max(torch.abs(q_r_exact - q_r_readout)).item())

        # 2. Intrinsic state-matching reward r_z(y):
        #    Exact one-step reward whose discounted sum equals F(s,a,z)^T z is r_z*(y) = B(y)^T Sigma_B^{-1} z.
        y_states = b_samples[:n_queries]
        r_exact = ((sigma_inv_true @ y_states.transpose(0, 1)).transpose(0, 1) * z_queries).sum(dim=-1)
        r_unwhitened = tracker.intrinsic_reward(y_states, z_queries, whitened=False)
        r_whitened = tracker.intrinsic_reward(y_states, z_queries, whitened=True)

        mse_unwhitened = float(torch.mean((r_unwhitened - r_exact).pow(2)).item())
        mse_whitened = float(torch.mean((r_whitened - r_exact).pow(2)).item())

        cos_unwhitened = float(torch.nn.functional.cosine_similarity(r_unwhitened, r_exact, dim=0).item())
        cos_whitened = float(torch.nn.functional.cosine_similarity(r_whitened, r_exact, dim=0).item())

        # 3. Single-state goal direction cosine alignment:
        #    Suppose we want policy gradient in action space where Jacobian d Q_B / d a = J_a.
        #    True state-matching direction in B-space is Sigma_B^{-1} B(g), whereas raw prompt uses B(g).
        g_states = b_samples[n_queries : 2 * n_queries]
        exact_goal_dir = (sigma_inv_true @ g_states.transpose(0, 1)).transpose(0, 1)
        whitened_goal_dir = tracker.whiten_vector(g_states)
        pg_cos_unwhitened = float(torch.nn.functional.cosine_similarity(g_states, exact_goal_dir, dim=-1).mean().item())
        pg_cos_whitened = float(torch.nn.functional.cosine_similarity(whitened_goal_dir, exact_goal_dir, dim=-1).mean().item())

        results_by_dim[f"dz_{d_z}_m_{m_dim}"] = {
            "d_z": d_z,
            "motion_dim": m_dim,
            "sphere_norm_mean": float(torch.linalg.vector_norm(b_samples, dim=-1).mean().item()),
            "spectral": spec,
            "linear_reward_readout_cancellation_max_error": linear_readout_max_err,
            "intrinsic_reward_mse_unwhitened": mse_unwhitened,
            "intrinsic_reward_mse_whitened": mse_whitened,
            "intrinsic_reward_mse_reduction_ratio": mse_unwhitened / max(mse_whitened, 1e-12),
            "intrinsic_reward_cosine_unwhitened": cos_unwhitened,
            "intrinsic_reward_cosine_whitened": cos_whitened,
            "goal_gradient_cosine_unwhitened": pg_cos_unwhitened,
            "goal_gradient_cosine_whitened": pg_cos_whitened,
        }

    r256 = results_by_dim["dz_256_m_229"]
    verdict_pass = (
        abs(r256["spectral"]["trace"] - 256.0) < 1e-2
        and r256["spectral"]["condition_number"] > 10.0
        and r256["linear_reward_readout_cancellation_max_error"] < 1e-3
        and r256["intrinsic_reward_mse_whitened"] < 0.15 * r256["intrinsic_reward_mse_unwhitened"]
        and r256["goal_gradient_cosine_whitened"] > 0.95
    )

    return {
        "experiment_id": "B01_GRAM_CONDITIONING_AND_WHITENING",
        "device": str(device),
        "elapsed_sec": round(time.time() - t0, 4),
        "verdict": "PASS" if verdict_pass else "FAIL",
        "results": results_by_dim,
    }


if __name__ == "__main__":
    out = run_b01_experiment()
    print(json.dumps(out, indent=2))


In [ ]:
%%writefile run_b02_nonstationarity_spike.py
"""Experiment Suite B02: Non-Stationarity, Batch-Norm Ordering, and Off-Policy Credit Assignment.

Tests:
1. UFO Batch Normalizer update ordering invariance:
   - Single-pass update on (obs, m_current) and (next_obs, m_next), then freeze BN (`eval`),
     proving expert queries and `z_update` goal relabeling cause ZERO normalizer drift.
   - Contrasts with the buggy double-`m_next` update and un-frozen expert query drift.
2. Strict separation of `behavior_z` vs `z_update` in replay.
3. Behavioral drift Delta_pi(z) and encoder drift Delta_B(m) across TaskReplay generations,
   plus generation-lag weighting and full-prefix zero-hidden RNN burn-in vs stale hidden states.
"""

from __future__ import annotations
import copy
import json
import time
from typing import Dict, Any
import torch
from fb_core import (
    BackwardMapB,
    ConditionedLowLevelActor,
    RecurrentLatentPlanner,
    RecurrentTaskTwinCritic,
    RunningMeanStdNormalizer,
    sample_ufo_mixed_z,
)


def run_b02_experiment(device_str: str = "auto", seed: int = 20261023) -> Dict[str, Any]:
    device = torch.device("cuda" if (device_str == "auto" and torch.cuda.is_available()) else ("cpu" if device_str == "auto" else device_str))
    torch.manual_seed(seed)
    t0 = time.time()

    d_z = 256
    m_dim = 229
    obs_dim = 96
    act_dim = 29
    batch_size = 128

    # 1. Test BN update ordering and freeze invariance
    bn_clean = RunningMeanStdNormalizer((m_dim,)).to(device)
    bn_buggy = RunningMeanStdNormalizer((m_dim,)).to(device)
    b_net = BackwardMapB(m_dim, d_z).to(device)

    m_curr = torch.randn(batch_size, m_dim, device=device) * 1.5 + 0.3
    m_next = torch.randn(batch_size, m_dim, device=device) * 2.2 - 0.4
    m_expert = torch.randn(batch_size, m_dim, device=device) * 5.0 + 3.0

    # Clean R2.1 UFO order: update once on m_curr, once on m_next, then freeze
    bn_clean.train()
    bn_clean.update_stats(m_curr)
    bn_clean.update_stats(m_next)
    bn_clean.eval()
    mean_before_queries = bn_clean.mean.clone()
    var_before_queries = bn_clean.var.clone()

    # Run z_update sampling and expert encoding under frozen BN
    behavior_z = torch.randn(batch_size, d_z, device=device)
    behavior_z_snapshot = behavior_z.clone()
    z_update, _ = sample_ufo_mixed_z(b_net, bn_clean(m_next), goal_ratio=0.5)
    _ = b_net(bn_clean(m_expert))

    bn_drift_under_frozen_queries = float(
        (torch.max(torch.abs(bn_clean.mean - mean_before_queries))
         + torch.max(torch.abs(bn_clean.var - var_before_queries))).item()
    )
    behavior_z_mutated = float(torch.max(torch.abs(behavior_z - behavior_z_snapshot)).item())

    # Buggy order: updates twice on m_next and updates on expert queries
    bn_buggy.train()
    bn_buggy.update_stats(m_curr)
    bn_buggy.update_stats(m_next)
    bn_buggy.update_stats(m_next)
    bn_buggy.update_stats(m_expert)
    buggy_mean_bias = float(torch.linalg.vector_norm(bn_buggy.mean - bn_clean.mean).item())

    # 2. Measure behavioral drift Delta_pi(z) and encoder drift Delta_B(m) across training steps
    actor = ConditionedLowLevelActor(obs_dim, act_dim, d_z).to(device)
    actor_init = copy.deepcopy(actor)
    b_init = copy.deepcopy(b_net)
    opt = torch.optim.Adam(list(actor.parameters()) + list(b_net.parameters()), lr=3e-3)

    obs_probe = torch.randn(batch_size, obs_dim, device=device)
    m_probe = bn_clean(m_curr)
    with torch.no_grad():
        z_probe_init = b_init(m_probe)
        a_probe_init = actor_init(obs_probe, z_probe_init)

    drift_trace = []
    for step in range(1, 21):
        opt.zero_grad()
        z_cur = b_net(m_probe)
        a_cur = actor(obs_probe, z_cur)
        # Synthetic representation evolution step
        loss = (a_cur.pow(2).mean()) + 0.1 * (z_cur[:, :32].pow(2).mean())
        loss.backward()
        opt.step()
        if step in (5, 10, 20):
            with torch.no_grad():
                z_now = b_net(m_probe)
                a_now = actor(obs_probe, z_probe_init)
                delta_b = float((1.0 - (z_now * z_probe_init).sum(dim=-1) / float(d_z)).mean().item())
                delta_pi = float(torch.linalg.vector_norm(a_now - a_probe_init, dim=-1).mean().item())
                drift_trace.append({"step": step, "delta_B_cosine": delta_b, "delta_pi_l2": delta_pi})

    # 3. Full-prefix zero-hidden RNN burn-in vs stale cached hidden state
    critic = RecurrentTaskTwinCritic(obs_q_dim=obs_dim, cmd_dim=d_z, hidden_dim=64).to(device)
    seq_len = 24
    burn_in = 12
    obs_seq = torch.randn(32, seq_len, obs_dim, device=device)
    cmd_seq = torch.randn(32, seq_len, d_z, device=device)

    # Stale hidden state captured before critic parameter update
    with torch.no_grad():
        h0 = torch.zeros(2, 32, 64, device=device)
        c0 = torch.zeros(2, 32, 64, device=device)
        _, (h_stale, c_stale) = critic.rnn1(obs_seq[:, :burn_in, :], (h0, c0))

    # Perturb critic weights to simulate online updates between rollout and replay sample
    with torch.no_grad():
        for p in critic.parameters():
            p.add_(0.15 * torch.randn_like(p))

    # Exact full-sequence reference from t=0 with current weights
    with torch.no_grad():
        q1_exact_full, _ = critic.forward_sequence(obs_seq, cmd_seq, burn_in_len=0)
        q1_exact_tail = q1_exact_full[:, burn_in:]

    # Burn-in reconstructed tail
    q1_burnin_tail, _ = critic.forward_sequence(obs_seq, cmd_seq, burn_in_len=burn_in)
    burnin_max_err = float(torch.max(torch.abs(q1_burnin_tail - q1_exact_tail)).item())

    # Stale cached hidden state tail
    out_stale, _ = critic.rnn1(obs_seq[:, burn_in:, :], (h_stale, c_stale))
    q1_stale_tail = critic.q1_head(torch.cat([out_stale, cmd_seq[:, burn_in:, :]], dim=-1)).squeeze(-1)
    stale_hidden_err = float(torch.max(torch.abs(q1_stale_tail - q1_exact_tail)).item())

    verdict_pass = (
        bn_drift_under_frozen_queries == 0.0
        and int(bn_clean.update_calls.item()) == 2
        and behavior_z_mutated == 0.0
        and buggy_mean_bias > 0.1
        and burnin_max_err < 1e-5
        and stale_hidden_err > 1e-2
    )

    return {
        "experiment_id": "B02_NONSTATIONARITY_BN_AND_CREDIT_ASSIGNMENT",
        "device": str(device),
        "elapsed_sec": round(time.time() - t0, 4),
        "verdict": "PASS" if verdict_pass else "FAIL",
        "bn_clean_update_calls": int(bn_clean.update_calls.item()),
        "bn_drift_under_frozen_queries": bn_drift_under_frozen_queries,
        "behavior_z_mutation_max_diff": behavior_z_mutated,
        "buggy_double_update_and_expert_mean_bias": buggy_mean_bias,
        "representation_drift_trace": drift_trace,
        "rnn_burnin_vs_exact_max_error": burnin_max_err,
        "rnn_stale_cached_hidden_max_error": stale_hidden_err,
    }


if __name__ == "__main__":
    print(json.dumps(run_b02_experiment(), indent=2))


In [ ]:
%%writefile run_b03_contact_aliasing_spike.py
"""Experiment Suite B03: Motion-Only Backward Map Aliasing and Perceptive Compensation.

Tests:
1. Kinematic-only B(m) aliasing: identical body kinematics m_t under three distinct physical
   contact regimes (free air swing, 0.40 m box hand/foot support, 0.60 m front-face impact)
   produce identical B(m_t) embeddings (zero B-space separation).
2. Perceptive compensation in F(o_F, a, z) and pi_theta(o_low, z): when terrain scan / ball
   state is provided in o_F and o_low, F and pi_theta disambiguate the transition outcome for a
   kinematically distinct target posture m_{t+1}^*.
3. Irreducible multi-force aliasing bottleneck: when two tasks require DIFFERENT contact impulses
   (e.g., light touch vs hard push-off) while holding the EXACT SAME kinematic trajectory m_{t+1}
   at the same obstacle geometry o_low, a purely kinematic z = B(m) cannot parameterize the force
   split unless planner P_psi uses non-local kinematic lead targets or B includes causal load response.
"""

from __future__ import annotations
import json
import time
from typing import Dict, Any
import torch
import torch.nn as nn
from fb_core import BackwardMapB, ForwardEnsembleF, ConditionedLowLevelActor


def run_b03_experiment(device_str: str = "auto", seed: int = 20261023) -> Dict[str, Any]:
    device = torch.device("cuda" if (device_str == "auto" and torch.cuda.is_available()) else ("cpu" if device_str == "auto" else device_str))
    torch.manual_seed(seed)
    t0 = time.time()

    m_dim = 229
    context_dim = 32  # terrain scan / ball state in o_low and o_F
    obs_dim = m_dim + context_dim
    act_dim = 29
    d_z = 256
    n_batch = 256

    b_net = BackwardMapB(m_dim, d_z).to(device)
    f_ens = ForwardEnsembleF(obs_dim, act_dim, d_z).to(device)
    actor = ConditionedLowLevelActor(obs_dim, act_dim, d_z).to(device)

    # 1. Identical kinematic posture m_same across 3 distinct contact contexts
    m_same = torch.randn(n_batch, m_dim, device=device)
    ctx_air = torch.zeros(n_batch, context_dim, device=device)
    ctx_box40 = torch.ones(n_batch, context_dim, device=device) * 0.40
    ctx_box60_impact = torch.ones(n_batch, context_dim, device=device) * (-0.60)

    with torch.no_grad():
        z_air = b_net(m_same)
        z_box40 = b_net(m_same)
        z_box60 = b_net(m_same)
        b_aliasing_max_diff = float(
            max(
                torch.max(torch.abs(z_air - z_box40)).item(),
                torch.max(torch.abs(z_air - z_box60)).item(),
            )
        )

    # 2. Train perceptive actor pi(o_low, z) to achieve distinct actions required in each context
    #    to reach the same kinematic outcome z_target = B(m_goal)
    m_goal = torch.randn(n_batch, m_dim, device=device)
    with torch.no_grad():
        z_goal = b_net(m_goal)

    obs_air = torch.cat([m_same, ctx_air], dim=-1)
    obs_box40 = torch.cat([m_same, ctx_box40], dim=-1)
    target_act_air = torch.tanh(m_same[:, :act_dim] * 0.5 + 0.4)
    target_act_box40 = torch.tanh(m_same[:, :act_dim] * 0.5 - 0.6)

    opt_perceptive = torch.optim.Adam(actor.parameters(), lr=3e-3)
    for _ in range(120):
        opt_perceptive.zero_grad()
        pred_air = actor(obs_air, z_goal)
        pred_box = actor(obs_box40, z_goal)
        loss = (pred_air - target_act_air).pow(2).mean() + (pred_box - target_act_box40).pow(2).mean()
        loss.backward()
        opt_perceptive.step()

    perceptive_mse = float(loss.item())

    # 3. Irreducible force-aliasing regime: SAME current obs (m_same, ctx_box40) AND SAME next
    #    kinematic frame m_next_same, but two different required contact forces (e.g., Force Mode 1 vs 2)
    #    Because B(m_next_same) produces a single z_kin, pi(obs_box40, z_kin) is a deterministic function
    #    of (obs_box40, z_kin) and cannot separate Force Mode 1 from Force Mode 2 without distinct z!
    force_target_1 = torch.ones(n_batch, act_dim, device=device) * 0.75
    force_target_2 = -torch.ones(n_batch, act_dim, device=device) * 0.75
    with torch.no_grad():
        pred_single_z = actor(obs_box40, z_goal)
        irreducible_same_z_mse = float(
            0.5 * ((pred_single_z - force_target_1).pow(2).mean() + (pred_single_z - force_target_2).pow(2).mean()).item()
        )

    verdict_pass = (
        b_aliasing_max_diff == 0.0
        and perceptive_mse < 0.01
        and irreducible_same_z_mse > 0.40
    )

    return {
        "experiment_id": "B03_CONTACT_ALIASING_AND_PERCEPTIVE_COMPENSATION",
        "device": str(device),
        "elapsed_sec": round(time.time() - t0, 4),
        "verdict": "PASS" if verdict_pass else "FAIL",
        "b_space_kinematic_aliasing_max_diff": b_aliasing_max_diff,
        "perceptive_context_disambiguation_action_mse": perceptive_mse,
        "irreducible_isokinematic_multi_force_mse": irreducible_same_z_mse,
    }


if __name__ == "__main__":
    print(json.dumps(run_b03_experiment(), indent=2))


In [ ]:
%%writefile run_b04_four_arm_ablation.py
"""Experiment Suite B04: Controlled 4-Arm Ablation — Pathwise World-Model Gradients vs FB Successor Derivatives.

Evaluates on the exact same DifferentiableContactPlant (causal state-action inputs only, ZERO future reference lookahead):
- Ground Truth Oracle: High-sample true-physics closed-loop trajectory return gradient d J_true / d theta.
- Arm A (Single-Task World Model, H=32): Learned world model W_psi(s_t, a_t) + task critic V_phi, unrolled H=32 steps.
- Arm B (Multi-Task World Model, H=32): W_psi(s_t, a_t) + z-conditioned policy pi_theta(s,z) and V_phi(s,z), H=32 steps.
- Arm C (Multi-Task World Model, H=1 First-Action Derivative Only): Same as Arm B, but unrolled H=1 step with stop-gradient
  on future policy calls inside V_phi(s_1, z), isolating H=1 vs H=32 pathwise differentiation.
- Arm D (Hybrid FB + Short-Horizon World Model, H in {1, 4, 8, 16, 32}):
  Unrolls W_psi(s_t, a_t) for H steps accumulating whitened intrinsic rewards r_z(hat_s_{t+1}) = B(hat_m_{t+1})^T W_lambda z
  plus terminal FB value bootstrap gamma^H F(hat_s_H, pi_theta(hat_s_H, z), z)^T z.
"""

from __future__ import annotations
import copy
import json
import math
import time
from typing import Dict, Any, List
import torch
import torch.nn as nn
from fb_core import BackwardMapB, ForwardEnsembleF, ConditionedLowLevelActor, EMAWhiteningTracker, project_to_sphere
from physics_env import DifferentiableContactPlant


class CausalWorldModel(nn.Module):
    """Causal state-delta world model W_psi(s_t, a_t) -> (s_{t+1}, m_{t+1}) with zero future reference hints."""

    def __init__(self, state_dim: int = 32, action_dim: int = 12, motion_dim: int = 229, hidden_dim: int = 256):
        super().__init__()
        self.delta_net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, state_dim),
        )
        self.motion_head = nn.Linear(state_dim, motion_dim)

    def step(self, s: torch.Tensor, a: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        s_next = s + 0.05 * self.delta_net(torch.cat([s, a], dim=-1))
        m_next = torch.tanh(self.motion_head(s_next))
        return s_next, m_next


def flatten_grad(params: List[nn.Parameter]) -> torch.Tensor:
    grads = []
    for p in params:
        if p.grad is not None:
            grads.append(p.grad.detach().reshape(-1))
        else:
            grads.append(torch.zeros_like(p).reshape(-1))
    return torch.cat(grads, dim=0)


def run_b04_experiment(device_str: str = "auto", seed: int = 20261023) -> Dict[str, Any]:
    device = torch.device("cuda" if (device_str == "auto" and torch.cuda.is_available()) else ("cpu" if device_str == "auto" else device_str))
    torch.manual_seed(seed)
    t0 = time.time()

    state_dim = 32
    act_dim = 12
    m_dim = 229
    d_z = 64
    gamma = 0.96
    batch_size = 128

    plant = DifferentiableContactPlant(state_dim=state_dim, action_dim=act_dim, motion_dim=m_dim).to(device)
    wm = CausalWorldModel(state_dim=state_dim, action_dim=act_dim, motion_dim=m_dim).to(device)
    b_net = BackwardMapB(motion_dim=m_dim, d_z=d_z, hidden_dim=128).to(device)
    f_ens = ForwardEnsembleF(obs_f_dim=state_dim, action_dim=act_dim, d_z=d_z, hidden_dim=128).to(device)
    actor = ConditionedLowLevelActor(obs_low_dim=state_dim, action_dim=act_dim, d_z=d_z, hidden_dim=128).to(device)
    whitener = EMAWhiteningTracker(d_z=d_z, beta=0.1, lambda_reg=1e-2).to(device)

    # 1. Collect real transitions and train CausalWorldModel (8-step rollout loss) + FB representation
    s0 = torch.randn(batch_size, state_dim, device=device) * 0.5
    z_batch = project_to_sphere(torch.randn(batch_size, d_z, device=device), d_z)

    opt_wm = torch.optim.Adam(wm.parameters(), lr=3e-3)
    for _ in range(80):
        opt_wm.zero_grad()
        s_real = s0
        s_hat = s0
        wm_loss = torch.tensor(0.0, device=device)
        for h in range(8):
            with torch.no_grad():
                a_h = actor(s_real, z_batch)
                s_real_next, m_real_next = plant.step(s_real, a_h)
            s_hat_next, m_hat_next = wm.step(s_hat, a_h)
            wm_loss = wm_loss + (s_hat_next - s_real_next).pow(2).mean() + 0.5 * (m_hat_next - m_real_next).pow(2).mean()
            s_real = s_real_next
            s_hat = s_hat_next
        wm_loss.backward()
        opt_wm.step()

    # Update whitener on real motion states
    with torch.no_grad():
        _, m_sample = plant.step(s0, actor(s0, z_batch))
        b_sample = b_net(m_sample)
        whitener.update(b_sample)

    # Fit F_ens to multi-step oracle return on current policy so F represents discounted future B visitation
    opt_f = torch.optim.Adam(f_ens.parameters(), lr=3e-3)
    for _ in range(100):
        opt_f.zero_grad()
        with torch.no_grad():
            s_cur = s0
            a0_cur = actor(s0, z_batch)
            s_cur, m_cur = plant.step(s_cur, a0_cur)
            phi_sum = whitener.whiten_vector(b_net(m_cur))
            disc = gamma
            for _ in range(15):
                a_next = actor(s_cur, z_batch)
                s_cur, m_cur = plant.step(s_cur, a_next)
                phi_sum = phi_sum + disc * whitener.whiten_vector(b_net(m_cur))
                disc *= gamma
        a0_pred = actor(s0, z_batch).detach()
        f1, f2 = f_ens(s0, a0_pred, z_batch)
        loss_f = (f1 - phi_sum).pow(2).mean() + (f2 - phi_sum).pow(2).mean()
        loss_f.backward()
        opt_f.step()

    # 2. Compute Ground-Truth True-Physics Closed-Loop Policy Gradient (H_true = 24)
    actor_params = list(actor.parameters())
    actor.zero_grad()
    s_cur = s0
    j_true = torch.tensor(0.0, device=device)
    for t in range(24):
        a_t = actor(s_cur, z_batch)
        s_cur, m_cur = plant.step(s_cur, a_t)
        r_t = whitener.intrinsic_reward(b_net(m_cur), z_batch, whitened=True)
        j_true = j_true + (gamma ** t) * r_t.mean()
    (-j_true).backward()
    grad_true = flatten_grad(actor_params)

    # 3. Evaluate Hybrid FB + World-Model across horizons H in {1, 4, 8, 16, 32}
    #    Note H=0 corresponds to pure 1st-action FB derivative: -F(s_0, pi_theta(s_0, z), z)^T z
    horizon_metrics = {}

    # Pure FB first-action derivative (Standard FB)
    actor.zero_grad()
    a0 = actor(s0, z_batch)
    q_fb0 = f_ens.conservative_scalar_value(s0, a0, z_batch, z_batch)
    (-q_fb0.mean()).backward()
    grad_fb_first = flatten_grad(actor_params)
    cos_fb_first = float(torch.nn.functional.cosine_similarity(grad_fb_first, grad_true, dim=0).item())
    horizon_metrics["H_0_standard_FB_first_action"] = {
        "gradient_cosine_vs_true_physics": cos_fb_first,
        "relative_l2_grad_error": float(torch.linalg.vector_norm(grad_fb_first - grad_true).item() / (torch.linalg.vector_norm(grad_true).item() + 1e-8)),
    }

    for h_val in [1, 4, 8, 16, 32]:
        actor.zero_grad()
        s_hat = s0
        s_real_check = s0
        total_obj = torch.tensor(0.0, device=device)
        rollout_state_err = 0.0
        for t in range(h_val):
            a_t = actor(s_hat, z_batch)
            with torch.no_grad():
                s_real_check, _ = plant.step(s_real_check, a_t)
            s_hat, m_hat = wm.step(s_hat, a_t)
            rollout_state_err = float(torch.linalg.vector_norm(s_hat.detach() - s_real_check, dim=-1).mean().item())
            r_hat = whitener.intrinsic_reward(b_net(m_hat), z_batch, whitened=True)
            total_obj = total_obj + (gamma ** t) * r_hat.mean()
        # Terminal FB bootstrap at step H
        a_term = actor(s_hat, z_batch)
        q_tail = f_ens.conservative_scalar_value(s_hat, a_term, z_batch, z_batch)
        total_obj = total_obj + (gamma ** h_val) * q_tail.mean()
        (-total_obj).backward()
        grad_h = flatten_grad(actor_params)
        cos_h = float(torch.nn.functional.cosine_similarity(grad_h, grad_true, dim=0).item())
        horizon_metrics[f"H_{h_val}"] = {
            "horizon": h_val,
            "gradient_cosine_vs_true_physics": cos_h,
            "compounded_state_error_at_H": rollout_state_err,
            "relative_l2_grad_error": float(torch.linalg.vector_norm(grad_h - grad_true).item() / (torch.linalg.vector_norm(grad_true).item() + 1e-8)),
        }

    # Compare unwhitened vs whitened hybrid rollout at H=8
    actor.zero_grad()
    s_hat = s0
    unwhitened_obj = torch.tensor(0.0, device=device)
    for t in range(8):
        a_t = actor(s_hat, z_batch)
        s_hat, m_hat = wm.step(s_hat, a_t)
        r_unw = whitener.intrinsic_reward(b_net(m_hat), z_batch, whitened=False)
        unwhitened_obj = unwhitened_obj + (gamma ** t) * r_unw.mean()
    a_term = actor(s_hat, z_batch)
    unwhitened_obj = unwhitened_obj + (gamma ** 8) * f_ens.conservative_scalar_value(s_hat, a_term, z_batch, z_batch).mean()
    (-unwhitened_obj).backward()
    grad_h8_unw = flatten_grad(actor_params)
    cos_h8_unw = float(torch.nn.functional.cosine_similarity(grad_h8_unw, grad_true, dim=0).item())

    # Throughput benchmark: World Model MLP step vs MuJoCo step
    t_wm0 = time.perf_counter()
    with torch.no_grad():
        s_tmp = s0
        a_tmp = actor(s0, z_batch)
        for _ in range(200):
            s_tmp, _ = wm.step(s_tmp, a_tmp)
    wm_steps_per_sec = (200 * batch_size) / max(time.perf_counter() - t_wm0, 1e-6)

    best_intermediate_cos = max(horizon_metrics["H_4"]["gradient_cosine_vs_true_physics"], horizon_metrics["H_8"]["gradient_cosine_vs_true_physics"])
    verdict_pass = (
        best_intermediate_cos > cos_fb_first
        and horizon_metrics["H_32"]["compounded_state_error_at_H"] > horizon_metrics["H_8"]["compounded_state_error_at_H"]
    )

    return {
        "experiment_id": "B04_FOUR_ARM_PATHWISE_VS_FB_ABLATION",
        "device": str(device),
        "elapsed_sec": round(time.time() - t0, 4),
        "verdict": "PASS" if verdict_pass else "FAIL",
        "standard_fb_first_action_cosine": cos_fb_first,
        "best_short_horizon_hybrid_cosine": best_intermediate_cos,
        "h8_whitened_vs_unwhitened_cosine": {
            "whitened": horizon_metrics["H_8"]["gradient_cosine_vs_true_physics"],
            "unwhitened": cos_h8_unw,
        },
        "horizon_sweep": horizon_metrics,
        "batched_wm_steps_per_sec": round(wm_steps_per_sec, 1),
    }


if __name__ == "__main__":
    print(json.dumps(run_b04_experiment(), indent=2))


In [ ]:
%%writefile run_b05_r21_verification_suite.py
"""Experiment Suite B05: R2.1 Terminal Boundary (`severe_penetration` vs `invalid`) and Multi-Height Curriculum Verification.

Executes the 3-part R2.1 verification suite on real MuJoCo 3.17 (`dt_physics=0.005s`, 4 substeps):
1. Test 1 (Credible Failure Commit): Substep 1 penetration > 0.03 m with finite states/contacts across all 4 substeps
   latches `terminated=True`, `status='severe_penetration'`, `phi_next=0.0`, commits once, and continues evaluation panel.
2. Test 2 (Post-Penetration Corruption Override): Substep 1 penetration > 0.03 m followed by Substep 3 `NaN` state or
   contact buffer overflow (`ncon >= nconmax`) strictly overrides `severe_penetration` with `status='invalid'`,
   rejects replay/normalizer update, and does not increment committed step count.
3. Test 3 (Curriculum Controller & Quota Accounting): Verifies `PoolQuotaSchedule` 50/30/20 allocation on `4096` rows
   with zero integer rounding loss, two-panel `95%` Wilson lower bound >= 0.70 unlock gate, and >10% low-height regression hold.
"""

from __future__ import annotations
import json
import math
import time
from typing import Dict, Any, List
import numpy as np
from physics_env import MuJoCoObstacleTraversalR21Env, SubstepSnapshot, evaluate_r21_substep_state_machine


def wilson_lower_bound_95(successes: int, total: int, z: float = 1.959963984540054) -> float:
    if total <= 0:
        return 0.0
    phat = successes / float(total)
    denom = 1.0 + (z * z) / total
    center = phat + (z * z) / (2.0 * total)
    half = z * math.sqrt((phat * (1.0 - phat) + (z * z) / (4.0 * total)) / total)
    return max(0.0, (center - half) / denom)


def compute_pool_quota_4096(unlocked_heights: List[float], total_rows: int = 4096) -> Dict[str, int]:
    """Allocates 4096 control rows across unlocked heights per Baseline 05 §6.1 with exact remainder accounting."""
    heights = sorted(unlocked_heights)
    if len(heights) <= 2:
        base = total_rows // len(heights)
        rem = total_rows - base * len(heights)
        out = {f"{h:.2f}": base for h in heights}
        out[f"{heights[-1]:.2f}"] += rem
        return out

    lowest_two = heights[:2]
    frontier = heights[-1]
    intermediate = heights[2:-1]

    if len(intermediate) == 0:
        # Only 3 heights unlocked: 50% frontier, 50% split across lowest two (or 50/30/20 merged)
        q_front = int(round(0.50 * total_rows))
        rem_low = total_rows - q_front
        q_l0 = rem_low // 2
        q_l1 = rem_low - q_l0
        return {
            f"{lowest_two[0]:.2f}": q_l0,
            f"{lowest_two[1]:.2f}": q_l1,
            f"{frontier:.2f}": q_front,
        }

    q_front = int(round(0.50 * total_rows))
    q_low_total = int(round(0.20 * total_rows))
    q_inter_total = total_rows - q_front - q_low_total

    out: Dict[str, int] = {}
    out[f"{lowest_two[0]:.2f}"] = q_low_total // 2
    out[f"{lowest_two[1]:.2f}"] = q_low_total - (q_low_total // 2)

    inter_base = q_inter_total // len(intermediate)
    inter_rem = q_inter_total - inter_base * len(intermediate)
    for idx, h in enumerate(intermediate):
        out[f"{h:.2f}"] = inter_base + (1 if idx == len(intermediate) - 1 else 0) * inter_rem

    out[f"{frontier:.2f}"] = q_front
    assert sum(out.values()) == total_rows, f"Quota sum mismatch: {sum(out.values())} != {total_rows}"
    return out


def run_b05_experiment() -> Dict[str, Any]:
    t0 = time.time()
    env = MuJoCoObstacleTraversalR21Env(obstacle_height=0.20)
    m0 = env.reset(seed=42)

    # Test 1: Credible severe penetration on Substep 1, valid substeps 2..4
    act_zero = np.zeros(29, dtype=np.float32)
    outcome_test1 = env.step_control(act_zero, inject_penetration_substep=1, inject_nan_substep=None)

    # Test 2: Substep 1 penetration > 0.03 m followed by Substep 3 NaN corruption
    step_before_test2 = env.step_count
    outcome_test2 = env.step_control(act_zero, inject_penetration_substep=1, inject_nan_substep=3)
    step_after_test2 = env.step_count

    # Test 2b: Substep 2 penetration > 0.03 m followed by Substep 4 contact buffer overflow (ncon >= nconmax)
    substeps_overflow = [
        SubstepSnapshot(1, 0.002, True, True, 4, 64, False, True),
        SubstepSnapshot(2, 0.042, True, True, 8, 64, False, True),
        SubstepSnapshot(3, 0.015, True, True, 12, 64, False, True),
        SubstepSnapshot(4, 0.010, True, True, 64, 64, False, True),
    ]
    outcome_test2b = evaluate_r21_substep_state_machine(
        substeps=substeps_overflow,
        final_state_finite=True,
        final_schema_ok=True,
        raw_phi_next=1.25,
    )

    # Test 3: Curriculum Wilson lower bound and PoolQuotaSchedule invariants
    lb_78_of_100 = wilson_lower_bound_95(78, 100)
    lb_79_of_100 = wilson_lower_bound_95(79, 100)
    quota_2 = compute_pool_quota_4096([0.10, 0.20])
    quota_5 = compute_pool_quota_4096([0.10, 0.20, 0.30, 0.40, 0.50])
    quota_7 = compute_pool_quota_4096([0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70])

    verdict_pass = (
        m0.shape == (229,)
        and outcome_test1.status == "severe_penetration"
        and outcome_test1.terminated is True
        and outcome_test1.commit_to_replay is True
        and outcome_test1.phi_next == 0.0
        and len(outcome_test1.substep_trace) == 4
        and outcome_test2.status == "invalid"
        and outcome_test2.commit_to_replay is False
        and outcome_test2.update_normalizers is False
        and step_after_test2 == step_before_test2
        and outcome_test2b.status == "invalid"
        and outcome_test2b.offending_substep == 4
        and lb_78_of_100 < 0.70
        and lb_79_of_100 >= 0.70
        and sum(quota_2.values()) == 4096
        and sum(quota_5.values()) == 4096
        and sum(quota_7.values()) == 4096
    )

    return {
        "experiment_id": "B05_R21_TERMINAL_BOUNDARY_AND_CURRICULUM_SUITE",
        "elapsed_sec": round(time.time() - t0, 4),
        "verdict": "PASS" if verdict_pass else "FAIL",
        "motion_descriptor_shape": list(m0.shape),
        "test1_credible_penetration": {
            "status": outcome_test1.status,
            "terminated": outcome_test1.terminated,
            "commit_to_replay": outcome_test1.commit_to_replay,
            "phi_next": outcome_test1.phi_next,
            "substeps_checked": len(outcome_test1.substep_trace),
        },
        "test2_post_penetration_nan_override": {
            "status": outcome_test2.status,
            "commit_to_replay": outcome_test2.commit_to_replay,
            "update_normalizers": outcome_test2.update_normalizers,
            "step_counter_unchanged": (step_after_test2 == step_before_test2),
            "offending_substep": outcome_test2.offending_substep,
        },
        "test2b_post_penetration_buffer_overflow_override": {
            "status": outcome_test2b.status,
            "offending_substep": outcome_test2b.offending_substep,
        },
        "test3_curriculum_and_quota": {
            "wilson_lb95_78_of_100": round(lb_78_of_100, 6),
            "wilson_lb95_79_of_100": round(lb_79_of_100, 6),
            "min_successes_of_100_for_lb70": 79,
            "quota_2_heights": quota_2,
            "quota_5_heights": quota_5,
            "quota_7_heights": quota_7,
        },
    }


if __name__ == "__main__":
    print(json.dumps(run_b05_experiment(), indent=2))


In [ ]:
%%writefile run_all_experiments.py
"""Master Runner for B01-B05 GPU/CPU Simulation & Verification Suite.

Executes all five experiment suites (B01 through B05), records hardware/GPU telemetry,
and writes `colab_gpu_results.json`.
"""

from __future__ import annotations
import json
import platform
import time
import torch

from run_b01_gram_spike import run_b01_experiment
from run_b02_nonstationarity_spike import run_b02_experiment
from run_b03_contact_aliasing_spike import run_b03_experiment
from run_b04_four_arm_ablation import run_b04_experiment
from run_b05_r21_verification_suite import run_b05_experiment


def main(output_path: str = "colab_gpu_results.json") -> dict:
    t0 = time.time()
    device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    suites = [
        run_b01_experiment(),
        run_b02_experiment(),
        run_b03_experiment(),
        run_b04_experiment(),
        run_b05_experiment(),
    ]
    summary = {
        "program": "fb-beyond-imitation-acceleration",
        "timestamp_utc": int(time.time()),
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "device_name": device_name,
        "total_elapsed_sec": round(time.time() - t0, 4),
        "all_passed": all(s["verdict"] == "PASS" for s in suites),
        "suites": {s["experiment_id"]: s for s in suites},
    }
    with open(output_path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"Saved complete results to {output_path} | all_passed={summary['all_passed']}")
    return summary


if __name__ == "__main__":
    main()


In [ ]:
!python3 run_all_experiments.py
import json
with open("colab_gpu_results.json") as f:
    res = json.load(f)
print("GPU_DEVICE:", res.get("device_name"), "| CUDA:", res.get("cuda_available"), "| ALL_PASSED:", res.get("all_passed"))
print(json.dumps({k: v["verdict"] for k, v in res["suites"].items()}, indent=2))
try:
    from google.colab import files
    files.download("colab_gpu_results.json")
except Exception as e:
    print("Download skipped:", e)
